# Notebook 03 — Cluster Analysis

**Goal:** Identify user segments using KMeans and HDBSCAN on the behavioural feature matrix. Select the best model via silhouette/elbow analysis, evaluate cluster quality, and characterise each segment.

**Inputs:** `data/processed/user_features.csv`

**Outputs:**
- `data/processed/cluster_labels.csv` — user IDs with cluster assignments
- `outputs/figures/elbow.html` — KMeans model selection chart
- `outputs/figures/umap_clusters.html` — 2D UMAP scatter
- `outputs/figures/cluster_heatmap.html` — feature profile heatmap

In [ ]:
import sys
import logging
from pathlib import Path

# Locate project root regardless of how the notebook is launched
def _find_project_root() -> Path:
    for candidate in [Path.cwd()] + list(Path.cwd().parents):
        if (candidate / 'configs' / 'config.yaml').exists():
            return candidate
    return Path('/Users/pranatisadhu/SpotifyClusterAnalysis')  # fallback

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)


In [ ]:
from src.data.loader import load_config

cfg = load_config(str(PROJECT_ROOT / 'configs/config.yaml'))
cluster_cfg = cfg['clustering']

feature_matrix = pd.read_csv(str(PROJECT_ROOT / 'data/processed/user_features.csv'), index_col=0)
print(f'Feature matrix: {feature_matrix.shape[0]} users × {feature_matrix.shape[1]} features')

## 1. KMeans — Elbow & Silhouette Analysis

Sweep k from 3 to 10. PCA is applied first to remove multicollinearity.

In [ ]:
from src.clustering.pipeline import run_clustering_pipeline
from src.clustering.evaluation import elbow_data
from src.visualization.plots import plot_elbow, save_figure

# Run KMeans sweep
kmeans_result = run_clustering_pipeline(
    feature_matrix=feature_matrix,
    algorithm='kmeans',
    use_pca=True,
    use_umap_viz=True,
    config=cluster_cfg,
)

print(f'Best k: {kmeans_result.n_clusters}')
print(f'Silhouette: {kmeans_result.silhouette:.4f}')
print(f'Davies-Bouldin: {kmeans_result.davies_bouldin:.4f}')
print(f'Calinski-Harabasz: {kmeans_result.calinski_harabasz:.1f}')

In [ ]:
# The pipeline already did the sweep internally;
# re-run sweep explicitly to capture all inertia/silhouette values for elbow plot
from src.clustering.pipeline import preprocess, reduce_with_pca, cluster_kmeans

X_scaled, feature_names = preprocess(feature_matrix)
X_pca, pca_obj = reduce_with_pca(X_scaled)

k_min = cluster_cfg['kmeans']['n_clusters_range'][0]
k_max = cluster_cfg['kmeans']['n_clusters_range'][1]

_, best_k, sil_scores, inertias = cluster_kmeans(
    X_pca,
    k_range=(k_min, k_max),
    n_init=cluster_cfg['kmeans']['n_init'],
    random_state=cluster_cfg['kmeans']['random_state'],
)

elbow_df = elbow_data(inertias, sil_scores)
fig_elbow = plot_elbow(elbow_df)
save_figure(fig_elbow, str(PROJECT_ROOT / 'outputs/figures/elbow'))
fig_elbow.show()

## 2. HDBSCAN — Density-Based Clustering

In [ ]:
hdbscan_result = run_clustering_pipeline(
    feature_matrix=feature_matrix,
    algorithm='hdbscan',
    use_pca=True,
    use_umap_viz=True,
    config=cluster_cfg,
)

print(f'HDBSCAN clusters: {hdbscan_result.n_clusters}')
print(f'Noise points: {(hdbscan_result.labels == -1).sum()}')
print(f'Silhouette: {hdbscan_result.silhouette:.4f}')

## 3. Model Selection

Compare KMeans (best k) vs HDBSCAN on key metrics.

In [ ]:
comparison = pd.DataFrame([
    {
        'Algorithm': f'KMeans (k={kmeans_result.n_clusters})',
        'Clusters': kmeans_result.n_clusters,
        'Noise Points': 0,
        'Silhouette ↑': kmeans_result.silhouette,
        'Davies-Bouldin ↓': kmeans_result.davies_bouldin,
        'Calinski-Harabasz ↑': kmeans_result.calinski_harabasz,
    },
    {
        'Algorithm': 'HDBSCAN',
        'Clusters': hdbscan_result.n_clusters,
        'Noise Points': int((hdbscan_result.labels == -1).sum()),
        'Silhouette ↑': hdbscan_result.silhouette,
        'Davies-Bouldin ↓': hdbscan_result.davies_bouldin,
        'Calinski-Harabasz ↑': hdbscan_result.calinski_harabasz,
    },
])
comparison.set_index('Algorithm', inplace=True)
comparison.round(4)

In [ ]:
# Select best result (higher silhouette = better separation)
if kmeans_result.silhouette >= hdbscan_result.silhouette:
    best_result = kmeans_result
    print(f'Selected: KMeans k={kmeans_result.n_clusters}')
else:
    best_result = hdbscan_result
    print(f'Selected: HDBSCAN ({hdbscan_result.n_clusters} clusters)')

## 4. Cluster Stability (Bootstrap Silhouette)

In [ ]:
from src.clustering.evaluation import bootstrap_silhouette

mean_sil, std_sil = bootstrap_silhouette(
    best_result.feature_matrix_scaled,
    best_result.labels,
    n_bootstrap=50,
)
print(f'Bootstrap silhouette: {mean_sil:.4f} ± {std_sil:.4f}')

## 5. Cluster Profiling

In [ ]:
from src.clustering.evaluation import summarise_clusters, label_clusters, feature_importance

cluster_summary = summarise_clusters(feature_matrix, best_result.labels)
cluster_names = label_clusters(cluster_summary, feature_matrix, best_result.labels)

print('Cluster labels:')
for cid, name in cluster_names.items():
    n = (best_result.labels == cid).sum()
    print(f'  Cluster {cid} ({n} users): {name}')

In [ ]:
# Feature importance across clusters
imp_df = feature_importance(feature_matrix, best_result.labels)
print('Top 10 most discriminating features:')
print(imp_df.head(10).to_string(index=False))

## 6. Save Cluster Labels

In [ ]:
labels_df = pd.DataFrame({
    'userid': feature_matrix.index,
    'cluster': best_result.labels,
    'cluster_name': [cluster_names.get(c, str(c)) for c in best_result.labels],
})

if best_result.umap_coords is not None:
    labels_df['umap_x'] = best_result.umap_coords[:, 0]
    labels_df['umap_y'] = best_result.umap_coords[:, 1]

labels_df.to_csv(str(PROJECT_ROOT / 'data/processed/cluster_labels.csv'), index=False)
print(f'Cluster labels saved: {labels_df.shape}')
labels_df['cluster_name'].value_counts()

Proceed to **Notebook 04** for interactive visualisations and business insights.